# check tta layers diff

## import

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import argparse
import random

import numpy as np
import torch
import torch.backends.cudnn as cudnn

import lavis.tasks as tasks
from lavis.common.config import Config
from lavis.common.dist_utils import get_rank, init_distributed_mode
from lavis.common.logger import setup_logger
from lavis.common.optims import (
    LinearWarmupCosineLRScheduler,
    LinearWarmupStepLRScheduler,
)
from lavis.common.utils import now

# imports modules for registration
from lavis.datasets.builders import *
from lavis.models import *
from lavis.processors import *
from lavis.runners.runner_base import RunnerBase
from lavis.tasks import *


def setup_seeds(config):
    seed = config.run_cfg.seed + get_rank()

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    cudnn.benchmark = False
    cudnn.deterministic = True

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## model_origin

In [2]:
args = argparse.Namespace(
    is_tta=True,
    cfg_path="lavis/projects/blip2/eval/ret_coco_eval_tent_debug_experiments.yaml",
    options=[]
)

job_id = now()

cfg = Config(args)

init_distributed_mode(cfg.run_cfg)

setup_seeds(cfg)

# set after init_distributed_mode() to only log on master.
setup_logger()

cfg.pretty_print()


2025-05-30 17:30:08,890 [INFO] 
=====  Running Parameters    =====
2025-05-30 17:30:08,891 [INFO] {
    "batch_size_eval": 16,
    "batch_size_train": 16,
    "device": "cuda",
    "dist_url": "env://",
    "distributed": false,
    "evaluate": true,
    "k_test": 128,
    "num_workers": 4,
    "output_dir": "output/BLIP2/Retrieval_COCO_tent_debug_experiments",
    "seed": 42,
    "task": "retrieval",
    "test_splits": [
        "test"
    ],
    "train_splits": [
        "train"
    ],
    "use_dist_eval_sampler": false,
    "valid_splits": [
        "val"
    ],
    "world_size": 1
}
2025-05-30 17:30:08,891 [INFO] 
======  Dataset Attributes  ======
2025-05-30 17:30:08,891 [INFO] 
======== coco_retrieval =======
2025-05-30 17:30:08,892 [INFO] {
    "build_info": {
        "annotations": {
            "test": {
                "md5": "3ff34b0ef2db02d01c37399f6a2a6cd1",
                "storage": "coco/annotations/coco_karpathy_test.json",
                "url": "https://storage.googl

Not using distributed mode


In [3]:
task = tasks.setup_task(cfg)
datasets = task.build_datasets(cfg)
model = task.build_model(cfg)

# runner = RunnerBase(
#     cfg=cfg, job_id=job_id, task=task, model=model, datasets=datasets
# )
# runner.evaluate(skip_reload=True)
# runner.evaluate_tta(skip_reload=True, tta_cfg=cfg.config.tta)


2025-05-30 17:30:08,900 [INFO] Building datasets...


Using downloaded and verified file: /home/zhh/ssd/excute/deeplearning/projects/datasets/coco/annotations/coco_karpathy_train.json
Using downloaded and verified file: /home/zhh/ssd/excute/deeplearning/projects/datasets/coco/annotations/coco_karpathy_val.json
Using downloaded and verified file: /home/zhh/ssd/excute/deeplearning/projects/datasets/coco/annotations/coco_karpathy_test.json


2025-05-30 17:30:09,572 [WARNING] /home/zhh/miniconda3/envs/blip2/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

2025-05-30 17:30:21,427 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/eva_vit.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer

Position interpolate from 16x16 to 26x26


2025-05-30 17:30:47,119 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/base_model.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature

## modal_adapt

In [4]:
model_adapt = task.build_model(cfg)

state_dict = torch.load("debug/tta_model_model_state_dict_after_i2tadapt_debug.pth", map_location="cpu")
model_adapt.load_state_dict(state_dict, strict=True)

Position interpolate from 16x16 to 26x26


2025-05-30 17:31:24,808 [INFO] Missing keys []
2025-05-30 17:31:24,811 [INFO] load checkpoint from /home/zhh/ssd/excute/deeplearning/projects/checkpoints/blip2_finetune_coco.pth
2025-05-30 17:31:24,915 [WARNING] /tmp/ipykernel_27659/826773678.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't ha

<All keys matched successfully>

In [5]:
model_adapt_t2i = task.build_model(cfg)

state_dict_t2i = torch.load("debug/tta_model_model_state_dict_after_t2iadapt_debug.pth", map_location="cpu")
model_adapt_t2i.load_state_dict(state_dict_t2i, strict=True)

2025-05-30 17:31:29,174 [WARNING] /home/zhh/miniconda3/envs/blip2/lib/python3.11/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

2025-05-30 17:31:40,897 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/eva_vit.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer

Position interpolate from 16x16 to 26x26


2025-05-30 17:32:06,025 [WARNING] /home/zhh/ssd/excute/deeplearning/projects/TTA_MM_Retrieval/LAVIS/lavis/models/base_model.py:42: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature

<All keys matched successfully>

## show diff layers

In [6]:
i = 0
for k,v in model.named_parameters():
    print(k, v.shape)
    # print(v)
    print(type(v))
    i+=1

    if i==5:
        break

query_tokens torch.Size([1, 32, 768])
<class 'torch.nn.parameter.Parameter'>
temp torch.Size([])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.cls_token torch.Size([1, 1, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.pos_embed torch.Size([1, 677, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.patch_embed.proj.weight torch.Size([1408, 3, 14, 14])
<class 'torch.nn.parameter.Parameter'>


In [7]:
j = 0
# for k,v in state_dict.items():
for k,v in model_adapt.named_parameters():
    print(k, v.shape)
    # print(v)
    print(type(v))
    j+=1

    if j==5:
        break

query_tokens torch.Size([1, 32, 768])
<class 'torch.nn.parameter.Parameter'>
temp torch.Size([])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.cls_token torch.Size([1, 1, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.pos_embed torch.Size([1, 677, 1408])
<class 'torch.nn.parameter.Parameter'>
visual_encoder.patch_embed.proj.weight torch.Size([1408, 3, 14, 14])
<class 'torch.nn.parameter.Parameter'>


In [8]:
j = 0
name_list = []
for ((k,v),(k_adapt,v_adapt)) in zip(model.named_parameters(), model_adapt.named_parameters()):

    # print(k, v.shape)
    # print(k_adapt, v_adapt.shape)
    # print(v)
    # print(type(v))
    diff = (v - v_adapt).detach().numpy().sum()
    if diff != 0:
        print()
        print(k, k_adapt)
        print("DIFF of original & adapted : ")
        print(diff)
        name_list.append(k)

    # try:
    #     diff = (v - v_adapt).detach().numpy().sum()
    #     if diff != 0:
    #         print("DIFF of original & adapted")
    #         print(diff)
    # except:
    #     print("except while comparing original & adapted")
    #     print(v - v_adapt)

    j+=1
    # if j==5:
    #     break


Qformer.bert.embeddings.LayerNorm.weight Qformer.bert.embeddings.LayerNorm.weight
DIFF of original & adapted : 
-0.018151835

Qformer.bert.embeddings.LayerNorm.bias Qformer.bert.embeddings.LayerNorm.bias
DIFF of original & adapted : 
0.0187537

Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight
DIFF of original & adapted : 
-0.07646209

Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias
DIFF of original & adapted : 
0.005798539

Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight
DIFF of original & adapted : 
0.003659308

Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias
DIFF of original & adapted : 
0.022769786

Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight Qformer

In [9]:
[ name for name in name_list if "LayerNorm" not in name]

[]

In [10]:
len(name_list)

62

In [11]:
j = 0
name_list_t2i = []
for ((k,v),(k_adapt,v_adapt)) in zip(model.named_parameters(), model_adapt_t2i.named_parameters()):

    # print(k, v.shape)
    # print(k_adapt, v_adapt.shape)
    # print(v)
    # print(type(v))
    diff = (v - v_adapt).detach().numpy().sum()
    if diff != 0:
        # print()
        # print(k, k_adapt)
        # print("DIFF of original & adapted : ")
        # print(diff)
        name_list_t2i.append(k)

    # try:
    #     diff = (v - v_adapt).detach().numpy().sum()
    #     if diff != 0:
    #         print("DIFF of original & adapted")
    #         print(diff)
    # except:
    #     print("except while comparing original & adapted")
    #     print(v - v_adapt)

    j+=1
    # if j==5:
    #     break

print([ name for name in name_list_t2i if "LayerNorm" not in name])
print(len(name_list_t2i))
print(name_list_t2i)

[]
50
['Qformer.bert.embeddings.LayerNorm.weight', 'Qformer.bert.embeddings.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.2.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.2.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.3.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.3.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.3.output.LayerNorm.weight', '

## diff of layer_list between t2i & i2t

In [12]:
diff_elements = list(set(name_list) - set(name_list_t2i))
print(len(diff_elements))

print("\r\n layers in i2t, not in t2i: ")
for e in diff_elements:
    if e not in name_list_t2i:
        print(e)

36

 layers in i2t, not in t2i: 
Qformer.bert.encoder.layer.7.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.5.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias
Qformer.bert.encoder.layer.6.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.4.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.6.crossattention.output.LayerNorm.bias
Qformer.bert.encoder.layer.0.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.1.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.6.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.1.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.10.output_query.LayerNorm.bias
Qformer.bert.encoder.layer.2.crossattention.output.LayerNorm.bias
Qformer.bert.encoder.layer.8.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight
Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight
Qformer.bert.encoder.layer.2.output_query.LayerNorm.weight
Qfor

In [13]:
diff_elements = list(set(name_list_t2i) - set(name_list))
print(len(diff_elements))

print("\r\n layers in t2i, not in i2t: ")
for e in diff_elements:
    if e not in name_list:
        print(e)



24

 layers in t2i, not in i2t: 
Qformer.bert.encoder.layer.7.output.LayerNorm.weight
Qformer.bert.encoder.layer.4.output.LayerNorm.bias
Qformer.bert.encoder.layer.10.output.LayerNorm.weight
Qformer.bert.encoder.layer.1.output.LayerNorm.weight
Qformer.bert.encoder.layer.7.output.LayerNorm.bias
Qformer.bert.encoder.layer.0.output.LayerNorm.weight
Qformer.bert.encoder.layer.8.output.LayerNorm.weight
Qformer.bert.encoder.layer.5.output.LayerNorm.bias
Qformer.bert.encoder.layer.6.output.LayerNorm.weight
Qformer.bert.encoder.layer.4.output.LayerNorm.weight
Qformer.bert.encoder.layer.3.output.LayerNorm.bias
Qformer.bert.encoder.layer.9.output.LayerNorm.weight
Qformer.bert.encoder.layer.8.output.LayerNorm.bias
Qformer.bert.encoder.layer.10.output.LayerNorm.bias
Qformer.bert.encoder.layer.2.output.LayerNorm.bias
Qformer.bert.encoder.layer.11.output.LayerNorm.weight
Qformer.bert.encoder.layer.0.output.LayerNorm.bias
Qformer.bert.encoder.layer.2.output.LayerNorm.weight
Qformer.bert.encoder.layer

In [14]:
name_list_layer_norm = []
for ((k,v),(k_adapt,v_adapt)) in zip(model.named_parameters(), model_adapt_t2i.named_parameters()):
    if "LayerNorm" in k:
        name_list_layer_norm.append(k)

print(len(name_list_layer_norm))
print(name_list_layer_norm)

88
['Qformer.bert.embeddings.LayerNorm.weight', 'Qformer.bert.embeddings.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.attention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.attention.output

In [15]:
diff_elements_i2t_pretrain = list(set(name_list_layer_norm) - set(name_list))
print("all layer_norm list - i2t list :", len(diff_elements_i2t_pretrain))

print(diff_elements_i2t_pretrain)



all layer_norm list - i2t list : 26
['Qformer.bert.encoder.layer.7.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.4.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.10.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.7.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.8.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.6.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.5.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.4.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.3.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.9.output.LayerNorm.weight', 'Qformer.bert.encoder.layer.8.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.10.output.LayerNorm.bias', 'Qformer.cls.predictions.transform.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.11.output.LayerNorm.weight', 'Qformer.bert.encoder.laye

In [16]:
diff_elements_t2i_pretrain = list(set(name_list_layer_norm) - set(name_list_t2i))
print("all layer_norm list - t2i list :", len(diff_elements_t2i_pretrain))

print(diff_elements_t2i_pretrain)

all layer_norm list - t2i list : 38
['Qformer.bert.encoder.layer.7.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.5.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.6.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.4.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.6.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.6.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.1.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.10.output_query.LayerNorm.bias', 'Qformer.bert.encoder.layer.2.crossattention.output.LayerNorm.bias', 'Qformer.bert.encoder.layer.8.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.output_query.LayerNorm.weight', 'Qformer.bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'Qformer.bert.

In [ ]:
torch.cuda.empty_cache()

## conclusion

- state_dict来自于tent，由于仅在recall阶段进行tta，image和text的feature都由Qformer计算，但不进行cross-attention交互，且各自有不同的输出head；
- 当 i2t tta时，Qformer中与image_embed输出相关的output模块不参与adapt，而text_input需要经过init_hidden_states=None的cross-attention模块输出text_embed，故cross-attention参与了adapt；
- 当 t2i tta时，Qformer中与text_embed输出相关的output_query模块和cross-attention模块不参与adapt；


# analysis rerank performance

## load sims_matrix and score_i2t/t2i

In [59]:
import numpy as np

# Load the npy files
sims_i2t = np.load("debug/sims_matrix_debug.npy").transpose(1,0)
sims_t2i = np.load("debug/sims_matrix_debug.npy")
score_i2t = np.load("debug/score_matrix_i2t_debug.npy")
score_t2i = np.load("debug/score_matrix_t2i_debug.npy")

print("sims_i2t shape:", sims_i2t.shape)
print("sims_t2i shape:", sims_t2i.shape)
print("score_i2t shape:", score_i2t.shape)
print("score_t2i shape:", score_t2i.shape)

sims_i2t shape: (5000, 25010)
sims_t2i shape: (25010, 5000)
score_i2t shape: (5000, 25010)
score_t2i shape: (25010, 5000)


In [60]:
import numpy as np

# Calculate mean, max, min, median for sims_i2t
sims_i2t_mean = np.mean(sims_i2t)
sims_i2t_max = np.max(sims_i2t)
sims_i2t_min = np.min(sims_i2t)
sims_i2t_median = np.median(sims_i2t)

# Calculate mean, max, min, median for score_i2t
score_i2t_mean = np.mean(score_i2t)
score_i2t_max = np.max(score_i2t)
score_i2t_min = np.min(score_i2t)
score_i2t_median = np.median(score_i2t)

print(sims_i2t_mean, sims_i2t_max, sims_i2t_min, sims_i2t_median)
print(score_i2t_mean, score_i2t_max, score_i2t_min, score_i2t_median)

0.255471 0.6125096 0.13369465 0.25392264
-99.49673 6.178787 -100.0 -100.0


## i2t 计算topk分数

### recall

In [61]:
k = 128
## 按照sims without rerank排序
topk_idx_sims = np.argsort(-sims_i2t, axis=1)[:, :k]  # 按行取前 k 个索引
topk_sims = np.take_along_axis(sims_i2t, topk_idx_sims, axis=1)

topk_score = np.take_along_axis(score_i2t, topk_idx_sims, axis=1)
# 这里的topk_score是按照topk_sims的topk顺序进行排序的

# score = itm_score + topk_sim
itm_score = topk_score - topk_sims
# 这里的itm_score是按照topk_sims的topk顺序进行排序的

In [62]:
np.where(score_i2t > -100,1,0).sum() / 5000

128.0

In [63]:
topk_sims[0][:10], itm_score[0][:10], topk_score[0][:10]

(array([0.5554966 , 0.5444903 , 0.53652364, 0.5284449 , 0.5192349 ,
        0.5135941 , 0.5107927 , 0.49589097, 0.4883941 , 0.48527563],
       dtype=float32),
 array([ 3.5666833 ,  3.7053866 ,  1.9656584 ,  2.9167795 ,  0.724301  ,
         0.64915115,  0.27841288, -0.502455  , -0.1639854 , -0.06578484],
       dtype=float32),
 array([ 4.12218   ,  4.249877  ,  2.502182  ,  3.4452243 ,  1.2435359 ,
         1.1627452 ,  0.78920555, -0.00656402,  0.3244087 ,  0.41949078],
       dtype=float32))

In [64]:
itm_score.shape

(5000, 128)

In [65]:
print(np.mean(topk_sims),np.max(topk_sims),np.min(topk_sims),np.median(topk_sims))
print(np.mean(itm_score),np.max(itm_score),np.min(itm_score),np.median(itm_score))
print(np.mean(topk_score),np.max(topk_score),np.min(topk_score),np.median(topk_score))

0.4083039 0.6125096 0.29702246 0.4017635
-2.0393946 5.6112356 -100.456314 -1.7661903
-1.6310914 6.178787 -100.0 -1.3629978


In [66]:
topk_idx_sims[0][:10]

array([    4,     3,     1,     0,  5696, 13804, 13806, 23576, 11859,
           2])

### rerank

In [70]:
## 按照itm_score only rerank排序
topk_idx_itm_score = np.argsort(-(score_i2t -sims_i2t), axis=1)[:, :k]  # 按行取前 k 个索引
topk_itm_score = np.take_along_axis((score_i2t-sims_i2t), topk_idx_itm_score, axis=1)

In [71]:
topk_idx_itm_score[0][:10]

array([    3,     4,     0,     1,  5696, 13804, 13806, 15789, 23560,
       15790])

### recall + rerank

In [72]:
## 按照score recall + rerank排序
topk_idx_score = np.argsort(-score_i2t, axis=1)[:, :k]  # 按行取前 k 个索引
topk_score = np.take_along_axis(score_i2t, topk_idx_score, axis=1)


In [73]:
topk_idx_score[0,:10]

array([    3,     4,     0,     1,  5696, 13804, 13806, 15789, 23560,
       15790])

### 加载 test label 分别计算分数

In [74]:
i2t_label = datasets["coco_retrieval"]["test"].img2txt

In [75]:
import numpy as np

def calculate_recall(topk_idx_sims, i2t_label):
    total_samples = len(i2t_label)
    recall_at_1 = 0
    recall_at_5 = 0
    recall_at_10 = 0

    for i in range(total_samples):
        true_label = i2t_label[i]
        topk_indices = topk_idx_sims[i]
        r1_flag = False
        r5_flag = False
        r10_flag = False

        for label in true_label:
            if label in topk_indices[:1].tolist():
                r1_flag = True
            if label in topk_indices[:5].tolist():
                r5_flag = True
            if label in topk_indices[:10].tolist():
                r10_flag = True

        if r1_flag:
            recall_at_1 += 1
        if r5_flag:
            recall_at_5 += 1
        if r10_flag:
            recall_at_10 += 1

    recall_at_1 /= total_samples
    recall_at_5 /= total_samples
    recall_at_10 /= total_samples

    return recall_at_1, recall_at_5, recall_at_10

In [76]:
topk_idx_sims[0][:10].tolist()

[4, 3, 1, 0, 5696, 13804, 13806, 23576, 11859, 2]

In [77]:
i2t_label[0]

[0, 1, 2, 3, 4]

In [78]:
# 调用方法并输出结果
recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_sims, i2t_label)
print("recall")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_itm_score, i2t_label)
print("rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_score, i2t_label)
print("recall+rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall
    Recall@1: 0.7436 Recall@5: 0.9424 Recall@10: 0.9742

rerank
    Recall@1: 0.8552 Recall@5: 0.9694 Recall@10: 0.9844

recall+rerank
    Recall@1: 0.8542 Recall@5: 0.9702 Recall@10: 0.9848



## t2i 计算topk分数

### recall

In [80]:
k = 128
## 按照sims without rerank排序
topk_idx_sims = np.argsort(-sims_t2i, axis=1)[:, :k]  # 按行取前 k 个索引
topk_sims = np.take_along_axis(sims_t2i, topk_idx_sims, axis=1)

topk_score = np.take_along_axis(score_t2i, topk_idx_sims, axis=1)
# 这里的topk_score是按照topk_sims的topk顺序进行排序的

# score = itm_score + topk_sim
itm_score = topk_score - topk_sims
# 这里的itm_score是按照topk_sims的topk顺序进行排序的

In [97]:
np.where(score_t2i > -100,1,0).sum() / 25010

128.0

In [82]:
topk_sims[0][:10], itm_score[0][:10], topk_score[0][:10]

(array([0.5284449 , 0.40552175, 0.40464383, 0.39618874, 0.38629746,
        0.3814321 , 0.37906712, 0.37194595, 0.37108356, 0.3689754 ],
       dtype=float32),
 array([ 2.9167795, -5.800764 , -6.152039 , -5.662221 , -6.0254555,
        -5.8958883, -6.414954 , -5.880032 , -6.2413235, -5.3765373],
       dtype=float32),
 array([ 3.4452243, -5.395242 , -5.747395 , -5.266032 , -5.6391582,
        -5.5144563, -6.0358872, -5.508086 , -5.8702397, -5.0075617],
       dtype=float32))

In [83]:
itm_score.shape

(25010, 128)

In [84]:
print(np.mean(topk_sims),np.max(topk_sims),np.min(topk_sims),np.median(topk_sims))
print(np.mean(itm_score),np.max(itm_score),np.min(itm_score),np.median(itm_score))
print(np.mean(topk_score),np.max(topk_score),np.min(topk_score),np.median(topk_score))

0.3304176 0.6125096 0.22253025 0.3164438
-3.9862561 5.6112356 -100.26652 -4.2982965
-3.655837 6.178787 -100.0 -3.978891


In [85]:
topk_idx_sims[0][:10]

array([   0,  788, 2761, 4100, 2371,  513, 1485, 3971, 2762, 3156])

### rerank

In [88]:
## 按照itm_score only rerank排序
topk_idx_itm_score = np.argsort(-(score_t2i -sims_t2i), axis=1)[:, :k]  # 按行取前 k 个索引
topk_itm_score = np.take_along_axis((score_t2i-sims_t2i), topk_idx_itm_score, axis=1)

In [89]:
topk_idx_itm_score[0][:10]

array([   0, 4833,   58, 3433, 2369, 4713,   21,  708, 1212, 2887])

### recall + rerank

In [90]:
## 按照score recall + rerank排序
topk_idx_score = np.argsort(-score_t2i, axis=1)[:, :k]  # 按行取前 k 个索引
topk_score = np.take_along_axis(score_t2i, topk_idx_score, axis=1)


In [91]:
topk_idx_score[0,:10]

array([   0, 4833,   58, 3433,   21, 2369, 4713,  708, 1212, 4830])

### 加载 test label 分别计算分数

In [92]:
t2i_label = datasets["coco_retrieval"]["test"].txt2img

In [99]:
import numpy as np

def calculate_recall(topk_idx_sims, t2i_label):
    total_samples = len(t2i_label)
    recall_at_1 = 0
    recall_at_5 = 0
    recall_at_10 = 0

    for i in range(total_samples):
        true_label = t2i_label[i]
        topk_indices = topk_idx_sims[i]
        if true_label in topk_indices[:1].tolist():
            recall_at_1 += 1
        if true_label in topk_indices[:5].tolist():
            recall_at_5 += 1
        if true_label in topk_indices[:10].tolist():
            recall_at_10 += 1

    recall_at_1 /= total_samples
    recall_at_5 /= total_samples
    recall_at_10 /= total_samples

    return recall_at_1, recall_at_5, recall_at_10

In [100]:
topk_idx_sims[0][:10].tolist()

[0, 788, 2761, 4100, 2371, 513, 1485, 3971, 2762, 3156]

In [101]:
t2i_label[0]

0

In [102]:
# 调用方法并输出结果
recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_sims, t2i_label)
print("recall")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_itm_score, t2i_label)
print("rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall_at_1, recall_at_5, recall_at_10 = calculate_recall(topk_idx_score, t2i_label)
print("recall+rerank")
print("    Recall@1: {:.4f}".format(recall_at_1), "Recall@5: {:.4f}".format(recall_at_5), "Recall@10: {:.4f}".format(recall_at_10))
print()

recall
    Recall@1: 0.6351 Recall@5: 0.8608 Recall@10: 0.9185

rerank
    Recall@1: 0.6820 Recall@5: 0.8766 Recall@10: 0.9248

recall+rerank
    Recall@1: 0.6825 Recall@5: 0.8773 Recall@10: 0.9263



: 

## analysis

- 在cos_sim不除以temperature的情况下，cos_sim的取值范围为-1,1；而itm_score的取值范围为0,15以上；
- 二者量纲并不相同，而在blip和blip2中，作者的代码将两者相加，达成了使用cos_sim微调itm_score的效果；
- 根据metric在i2t中仅使用itm_score，recall@1会上升，但recall@5和10会下降；
- 根据metric在t2i中仅使用itm_score，recall@1,recall@5,recall@10均会下降；

# Label Smoothing Strategy

## ex 1 cos_sim

In [1]:
import torch
import numpy as np
import random

random_list = [random.uniform(0, 0.01) for _ in range(123)]
print(random_list)
topk_list = [0.51, 0.5, 0.45, 0.42, 0.41]
topk_list.extend(random_list)
print(topk_list)
topk_tensor = torch.Tensor(topk_list)
print(topk_tensor)

[0.003376642897601886, 0.009311357787734406, 0.006166205160655777, 0.00912117520348541, 0.009362594528506994, 0.004263873635041894, 0.00209553685665681, 0.0012004374867485956, 0.004914322366852532, 0.007564963560575005, 0.006972779962714656, 0.008033941886974616, 0.007886531931724738, 0.00430434628923694, 0.007424195071247126, 0.003839928860704898, 0.008949708747847805, 0.0063550873322637084, 0.0010195799113107684, 0.00493654831048915, 0.0037946141406723388, 2.957967637977732e-05, 0.004705433686504575, 0.007123165658266201, 0.0038852992374280816, 0.007109190607285269, 0.008108902714971474, 0.007157592709720459, 0.0001971289823960387, 0.0009055984801293438, 0.0012427285452202852, 0.0003730646843870389, 0.005745098801545948, 0.0020540092362569227, 0.006142175064605482, 0.006049205393197935, 0.004702540772904786, 0.009901441995377451, 0.0005877255385061209, 0.002337696068817785, 0.009278428838965657, 0.003109312682926706, 0.008090247831594392, 0.009453920848114925, 0.004226988995742355, 0

In [8]:
import torch.nn.functional as F
# print(F.softmax(topk_tensor, dim=0))
# print(F.log_softmax(topk_tensor, dim=0))
# print(-(F.softmax(topk_tensor, dim=0) * F.log_softmax(topk_tensor, dim=0)))
print(-(F.softmax(topk_tensor, dim=0) * F.log_softmax(topk_tensor, dim=0)).sum())

print(F.softmax(topk_tensor, dim=0))

tensor(4.8469)
tensor([0.0127, 0.0125, 0.0119, 0.0116, 0.0115, 0.0076, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0076, 0.0076, 0.0076, 0.0076, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0076, 0.0077, 0.0076, 0.0077, 0.0077, 0.0076, 0.0076, 0.0076, 0.0076,
        0.0076, 0.0077, 0.0076, 0.0077, 0.0077, 0.0077, 0.0076, 0.0076, 0.0076,
        0.0076, 0.0076, 0.0076, 0.0077, 0.0076, 0.0076, 0.0077, 0.0076, 0.0076,
        0.0077, 0.0076, 0.0077, 0.0077, 0.0076, 0.0077, 0.0076, 0.0076, 0.0076,
        0.0076, 0.0076, 0.0076, 0.0077, 0.0076, 0.0076, 0.0076, 0.0076, 0.0076,
        0.0076, 0.0076, 0.0076, 0.0077, 0.0076, 0.0076, 0.0076, 0.0077, 0.0076,
        0.0076, 0.0076, 0.0077, 0.0077, 0.0077, 0.0076, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0076, 0.0076, 0.0077, 0.0077, 0.0076, 0.0077, 0.0076, 0.0076,
        0.0076, 0.0077, 0.0076, 0.0076, 0.0076, 0.0077, 0.0077, 0.0077, 0.0076,
        0.0076, 0.0077, 0.0076, 0.0076, 0.0076, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0076, 0

In [ ]:
print(-(F.sigmoid(topk_tensor) * torch.log(F.sigmoid(topk_tensor))).sum())

print(F.sigmoid(topk_tensor) )

tensor(44.0828)
tensor([0.6248, 0.6225, 0.6106, 0.6035, 0.6011, 0.5008, 0.5023, 0.5015, 0.5023,
        0.5023, 0.5011, 0.5005, 0.5003, 0.5012, 0.5019, 0.5017, 0.5020, 0.5020,
        0.5011, 0.5019, 0.5010, 0.5022, 0.5016, 0.5003, 0.5012, 0.5009, 0.5000,
        0.5012, 0.5018, 0.5010, 0.5018, 0.5020, 0.5018, 0.5000, 0.5002, 0.5003,
        0.5001, 0.5014, 0.5005, 0.5015, 0.5015, 0.5012, 0.5025, 0.5001, 0.5006,
        0.5023, 0.5008, 0.5020, 0.5024, 0.5011, 0.5017, 0.5003, 0.5006, 0.5009,
        0.5007, 0.5004, 0.5012, 0.5019, 0.5013, 0.5004, 0.5011, 0.5008, 0.5005,
        0.5001, 0.5014, 0.5004, 0.5019, 0.5010, 0.5008, 0.5007, 0.5025, 0.5002,
        0.5014, 0.5004, 0.5017, 0.5023, 0.5016, 0.5004, 0.5020, 0.5016, 0.5023,
        0.5023, 0.5009, 0.5004, 0.5020, 0.5019, 0.5012, 0.5020, 0.5004, 0.5009,
        0.5008, 0.5024, 0.5000, 0.5012, 0.5005, 0.5017, 0.5018, 0.5020, 0.5004,
        0.5010, 0.5020, 0.5009, 0.5012, 0.5004, 0.5022, 0.5017, 0.5018, 0.5022,
        0.5025, 0.5006, 

## ex 2 cos_sim

In [11]:
# random_list = [random.uniform(0, 0.01) for _ in range(127)]
random_list = [0.1/127 for _ in range(127)]
print(random_list)
topk_list = [0.9]
topk_list.extend(random_list)
print(topk_list)
topk_tensor = torch.Tensor(topk_list)
print(topk_tensor)

[0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.0007874015748031497, 0.00078740

In [12]:
import math

In [13]:
print(F.softmax(topk_tensor, dim=0))
# print([ math.log(x) for x in F.softmax(topk_tensor, dim=0).numpy().tolist() ])
# print(F.log_softmax(topk_tensor, dim=0))
# print(-(F.softmax(topk_tensor, dim=0) * F.log_softmax(topk_tensor, dim=0)))
print(-(F.softmax(topk_tensor, dim=0) * F.log_softmax(topk_tensor, dim=0)).sum())

tensor([0.0190, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 

In [14]:
print(-(F.sigmoid(topk_tensor) * torch.log(F.sigmoid(topk_tensor))).sum())

print(F.sigmoid(topk_tensor) )

tensor(44.2497)
tensor([0.7109, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002, 0.5002,
        0.5002, 0.5002, 

In [24]:
print(F.sigmoid(torch.Tensor([0.9])) )

print(F.sigmoid(torch.Tensor([0.1/127])) )
print(F.sigmoid(torch.Tensor([-0.9])) )
print(F.sigmoid(torch.Tensor([-9])) )

tensor([0.7109])
tensor([0.5002])
tensor([0.2891])
tensor([0.0001])


## ex 3 score

In [35]:
# random_list = [random.uniform(0, 0.01) for _ in range(127)]
random_list = [-5 for _ in range(127)]
print(random_list)
topk_list = [4]
topk_list.extend(random_list)
print(topk_list)
topk_tensor = torch.Tensor(topk_list)
print(topk_tensor)

[-5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5]
[4, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5, -5,

In [36]:
print(F.softmax(topk_tensor, dim=0))
# print([ math.log(x) for x in F.softmax(score, dim=0).numpy().tolist() ])
# print(F.log_softmax(score, dim=0))
# print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)))
print(-(F.softmax(topk_tensor, dim=0) * F.log_softmax(topk_tensor, dim=0)).sum())

tensor([9.8457e-01, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04, 1.2151e-04,
        1.2151e-04, 1.2151e-04, 1.2151e-

In [37]:
print(-(F.sigmoid(topk_tensor) * torch.log(F.sigmoid(topk_tensor))).sum())

print(F.sigmoid(topk_tensor) )

tensor(4.2735)
tensor([0.9820, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067, 0.0067,
        0.0067, 0.0067, 0

## ex 4 score

In [34]:
score = torch.Tensor([ 2.5251,  1.7987,  2.4984,  2.5456,  0.3018,  0.6873,  1.4105,  0.7351,
         0.3194,  1.4600,  0.2504, -2.3080, -0.3669, -1.0338, -0.8816, -1.0167,
        -2.3506, -3.9693, -0.2150, -0.0291, -1.4228, -2.3420, -0.1471, -3.8148,
        -2.0076, -1.8196, -2.3271, -0.7736, -2.6617, -0.2045, -3.0147, -0.3503,
        -1.4831, -2.4310, -0.6229, -1.0497, -4.0194, -1.7258, -3.2078, -1.6475,
        -0.6561, -2.3149, -0.2777, -4.9612, -4.4866, -3.2148, -3.3607, -1.1115,
        -1.6387, -2.0795, -3.3493, -3.4117, -2.4388, -0.3735, -4.3452, -4.9870,
        -0.5091, -3.6897, -2.1280, -1.6361, -3.8723, -3.5618, -4.1496, -5.4335,
        -4.9748, -3.9281, -1.5872, -2.4761, -4.1328, -3.3151, -0.7322, -3.8610,
        -3.3226, -3.7666, -0.8437, -0.8367, -2.7714, -2.7362, -3.1571, -1.3180,
        -3.0055, -3.7977, -3.6444, -3.2574, -2.1181, -0.7243, -2.4106, -3.3575,
        -0.6314, -4.4725, -4.1300, -5.6244, -4.2784, -3.8402, -3.3613, -2.2780,
        -3.9022, -5.2376, -3.6194, -5.0524, -3.3743, -1.7079, -4.7432, -3.7393,
        -4.8307, -2.5210, -1.4971, -2.3383, -0.8525, -3.6789, -2.2030, -2.6994,
        -5.0277, -5.2465, -1.2598, -2.4906, -1.0846, -2.7455, -4.4144, -1.1020,
        -3.3042, -1.0451, -3.5574, -0.6343, -0.8737, -3.4680, -3.5093, -2.6475])

In [36]:
print(F.softmax(score, dim=0))
# print([ math.log(x) for x in F.softmax(score, dim=0).numpy().tolist() ])
# print(F.log_softmax(score, dim=0))
# print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)))
print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)).sum())

tensor([1.5444e-01, 7.4693e-02, 1.5037e-01, 1.5764e-01, 1.6718e-02, 2.4581e-02,
        5.0663e-02, 2.5785e-02, 1.7015e-02, 5.3234e-02, 1.5881e-02, 1.2296e-03,
        8.5659e-03, 4.3969e-03, 5.1197e-03, 4.4727e-03, 1.1783e-03, 2.3349e-04,
        9.9711e-03, 1.2008e-02, 2.9799e-03, 1.1885e-03, 1.0672e-02, 2.7250e-04,
        1.6605e-03, 2.0039e-03, 1.2063e-03, 5.7036e-03, 8.6329e-04, 1.0076e-02,
        6.0653e-04, 8.7093e-03, 2.8055e-03, 1.0873e-03, 6.6312e-03, 4.3275e-03,
        2.2208e-04, 2.2010e-03, 5.0002e-04, 2.3802e-03, 6.4147e-03, 1.2212e-03,
        9.3651e-03, 8.6596e-05, 1.3919e-04, 4.9653e-04, 4.2913e-04, 4.0682e-03,
        2.4013e-03, 1.5453e-03, 4.3405e-04, 4.0779e-04, 1.0788e-03, 8.5096e-03,
        1.6033e-04, 8.4390e-05, 7.4305e-03, 3.0882e-04, 1.4721e-03, 2.4075e-03,
        2.5728e-04, 3.5095e-04, 1.9497e-04, 5.3998e-05, 8.5426e-05, 2.4331e-04,
        2.5282e-03, 1.0393e-03, 1.9827e-04, 4.4915e-04, 5.9447e-03, 2.6020e-04,
        4.4579e-04, 2.8596e-04, 5.3174e-

In [41]:
print(F.sigmoid(score))
# print([ math.log(x) for x in F.softmax(score, dim=0).numpy().tolist() ])
# print(F.log_softmax(score, dim=0))
# print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)))
print(-(F.sigmoid(score) * torch.log(F.sigmoid(score))).sum())

tensor([0.9259, 0.8580, 0.9240, 0.9273, 0.5749, 0.6654, 0.8038, 0.6759, 0.5792,
        0.8115, 0.5623, 0.0905, 0.4093, 0.2623, 0.2928, 0.2657, 0.0870, 0.0185,
        0.4465, 0.4927, 0.1942, 0.0877, 0.4633, 0.0216, 0.1184, 0.1395, 0.0889,
        0.3157, 0.0653, 0.4491, 0.0468, 0.4133, 0.1850, 0.0808, 0.3491, 0.2593,
        0.0176, 0.1511, 0.0389, 0.1614, 0.3416, 0.0899, 0.4310, 0.0070, 0.0111,
        0.0386, 0.0335, 0.2476, 0.1626, 0.1111, 0.0339, 0.0319, 0.0803, 0.4077,
        0.0128, 0.0068, 0.3754, 0.0244, 0.1064, 0.1630, 0.0204, 0.0276, 0.0155,
        0.0043, 0.0069, 0.0193, 0.1698, 0.0776, 0.0158, 0.0351, 0.3247, 0.0206,
        0.0348, 0.0226, 0.3008, 0.3022, 0.0589, 0.0609, 0.0408, 0.2112, 0.0472,
        0.0219, 0.0255, 0.0371, 0.1074, 0.3264, 0.0824, 0.0337, 0.3472, 0.0113,
        0.0158, 0.0036, 0.0137, 0.0210, 0.0335, 0.0930, 0.0198, 0.0053, 0.0261,
        0.0064, 0.0331, 0.1534, 0.0086, 0.0232, 0.0079, 0.0744, 0.1829, 0.0880,
        0.2989, 0.0246, 0.0995, 0.0630, 

## ex 5 score

In [ ]:
score = torch.Tensor([ 0.1861, -0.1353,  0.0433,  0.2448,  0.2569,  0.1487, -0.0225,  0.0656,
        -0.1290, -0.1263, -0.1100,  0.0749,  0.1404,  0.0999,  0.0774,  0.7098,
         0.1034,  0.0222,  0.0464,  0.2748,  0.1946,  0.0283, -0.3403,  0.1891,
        -1.2059, -1.3221,  0.2691, -0.1907,  0.0470,  0.1387,  0.3835, -0.0047,
        -0.0721,  0.0430,  0.0228,  0.0412,  0.0233,  0.1505, -0.3700, -0.8372,
         0.1514,  0.0333, -0.0867, -0.7579, -0.2178,  0.0384, -0.3476, -0.5086,
         0.1748, -0.0286, -0.1647, -0.1223, -0.1980,  0.0598, -0.0923, -0.4152,
        -0.7515, -0.2440,  0.1311, -1.4587, -0.1334,  0.0222, -0.3817, -1.1041,
        -0.0721, -0.1133, -0.1733, -0.0421, -0.1024, -0.1077,  0.1378,  0.0955,
        -0.1986, -0.7618,  0.0479, -0.4493, -0.5466, -0.0626, -0.1593, -0.2145,
         0.1208, -1.6073, -0.0514, -1.1260, -0.0346,  0.0127, -0.1411, -0.1093,
        -0.1426, -0.0500, -0.3367, -0.2101, -0.4942, -0.1560, -0.3084, -0.2088,
        -0.3985, -1.3578, -0.2336, -0.1146, -0.0370, -0.0523, -0.1758, -0.0623,
         0.1343, -0.3647, -0.7058, -1.0140, -1.4076, -0.1413, -0.2043, -0.7042,
        -1.4981, -0.3609,  0.1634,  0.0144, -0.2519, -0.2833, -0.2898, -0.4575,
        -0.2307, -0.0288, -0.4821, -0.1848, -0.3248, -0.2738, -0.3510, -0.3737])

In [41]:
print(F.softmax(score, dim=0))
# print([ math.log(x) for x in F.softmax(score, dim=0).numpy().tolist() ])
# print(F.log_softmax(score, dim=0))
# print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)))
print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)).sum())

tensor([0.0108, 0.0079, 0.0094, 0.0115, 0.0116, 0.0104, 0.0088, 0.0096, 0.0079,
        0.0079, 0.0081, 0.0097, 0.0104, 0.0100, 0.0097, 0.0183, 0.0100, 0.0092,
        0.0094, 0.0119, 0.0109, 0.0093, 0.0064, 0.0109, 0.0027, 0.0024, 0.0118,
        0.0074, 0.0094, 0.0103, 0.0132, 0.0090, 0.0084, 0.0094, 0.0092, 0.0094,
        0.0092, 0.0105, 0.0062, 0.0039, 0.0105, 0.0093, 0.0083, 0.0042, 0.0072,
        0.0094, 0.0064, 0.0054, 0.0107, 0.0088, 0.0076, 0.0080, 0.0074, 0.0096,
        0.0082, 0.0059, 0.0042, 0.0071, 0.0103, 0.0021, 0.0079, 0.0092, 0.0061,
        0.0030, 0.0084, 0.0080, 0.0076, 0.0086, 0.0081, 0.0081, 0.0103, 0.0099,
        0.0074, 0.0042, 0.0094, 0.0057, 0.0052, 0.0085, 0.0077, 0.0073, 0.0102,
        0.0018, 0.0086, 0.0029, 0.0087, 0.0091, 0.0078, 0.0081, 0.0078, 0.0086,
        0.0064, 0.0073, 0.0055, 0.0077, 0.0066, 0.0073, 0.0060, 0.0023, 0.0071,
        0.0080, 0.0087, 0.0085, 0.0076, 0.0085, 0.0103, 0.0063, 0.0044, 0.0033,
        0.0022, 0.0078, 0.0073, 0.0045, 

In [42]:
print(F.sigmoid(score))
# print([ math.log(x) for x in F.softmax(score, dim=0).numpy().tolist() ])
# print(F.log_softmax(score, dim=0))
# print(-(F.softmax(score, dim=0) * F.log_softmax(score, dim=0)))
print(-(F.sigmoid(score) * torch.log(F.sigmoid(score))).sum())

tensor([0.5464, 0.4662, 0.5108, 0.5609, 0.5639, 0.5371, 0.4944, 0.5164, 0.4678,
        0.4685, 0.4725, 0.5187, 0.5350, 0.5250, 0.5193, 0.6704, 0.5258, 0.5055,
        0.5116, 0.5683, 0.5485, 0.5071, 0.4157, 0.5471, 0.2304, 0.2105, 0.5669,
        0.4525, 0.5117, 0.5346, 0.5947, 0.4988, 0.4820, 0.5107, 0.5057, 0.5103,
        0.5058, 0.5376, 0.4085, 0.3021, 0.5378, 0.5083, 0.4783, 0.3191, 0.4458,
        0.5096, 0.4140, 0.3755, 0.5436, 0.4929, 0.4589, 0.4695, 0.4507, 0.5149,
        0.4769, 0.3977, 0.3205, 0.4393, 0.5327, 0.1887, 0.4667, 0.5055, 0.4057,
        0.2490, 0.4820, 0.4717, 0.4568, 0.4895, 0.4744, 0.4731, 0.5344, 0.5239,
        0.4505, 0.3183, 0.5120, 0.3895, 0.3667, 0.4844, 0.4603, 0.4466, 0.5302,
        0.1670, 0.4872, 0.2449, 0.4914, 0.5032, 0.4648, 0.4727, 0.4644, 0.4875,
        0.4166, 0.4477, 0.3789, 0.4611, 0.4235, 0.4480, 0.4017, 0.2046, 0.4419,
        0.4714, 0.4908, 0.4869, 0.4562, 0.4844, 0.5335, 0.4098, 0.3305, 0.2662,
        0.1966, 0.4647, 0.4491, 0.3309, 

## ex 5 cos_sim

In [45]:
cos_sim = torch.Tensor([0.5292, 0.5214, 0.5199, 0.5109, 0.5083, 0.5016, 0.5011, 0.4996, 0.4989,
        0.4985, 0.4950, 0.4905, 0.4878, 0.4865, 0.4861, 0.4859, 0.4857, 0.4849,
        0.4849, 0.4843, 0.4841, 0.4833, 0.4830, 0.4827, 0.4822, 0.4820, 0.4816,
        0.4800, 0.4799, 0.4797, 0.4790, 0.4782, 0.4780, 0.4779, 0.4773, 0.4763,
        0.4756, 0.4756, 0.4746, 0.4736, 0.4735, 0.4735, 0.4728, 0.4719, 0.4694,
        0.4673, 0.4672, 0.4661, 0.4653, 0.4631, 0.4625, 0.4608, 0.4608, 0.4608,
        0.4600, 0.4592, 0.4590, 0.4589, 0.4581, 0.4577, 0.4573, 0.4570, 0.4567,
        0.4566, 0.4553, 0.4552, 0.4550, 0.4548, 0.4541, 0.4541, 0.4526, 0.4524,
        0.4518, 0.4512, 0.4510, 0.4500, 0.4499, 0.4497, 0.4496, 0.4493, 0.4489,
        0.4485, 0.4480, 0.4479, 0.4479, 0.4477, 0.4474, 0.4473, 0.4473, 0.4472,
        0.4461, 0.4459, 0.4442, 0.4442, 0.4441, 0.4439, 0.4437, 0.4430, 0.4421,
        0.4420, 0.4415, 0.4415, 0.4414, 0.4413, 0.4412, 0.4409, 0.4405, 0.4404,
        0.4402, 0.4402, 0.4399, 0.4389, 0.4381, 0.4379, 0.4375, 0.4374, 0.4365,
        0.4362, 0.4361, 0.4354, 0.4353, 0.4348, 0.4339, 0.4337, 0.4330, 0.4323,
        0.4323, 0.4321])

In [46]:
print(F.softmax(cos_sim, dim=0))
print(-(F.softmax(cos_sim, dim=0) * F.log_softmax(cos_sim, dim=0)).sum())

tensor([0.0084, 0.0083, 0.0083, 0.0082, 0.0082, 0.0081, 0.0081, 0.0081, 0.0081,
        0.0081, 0.0081, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080,
        0.0080, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080, 0.0080,
        0.0080, 0.0080, 0.0080, 0.0079, 0.0079, 0.0079, 0.0079, 0.0079, 0.0079,
        0.0079, 0.0079, 0.0079, 0.0079, 0.0079, 0.0079, 0.0079, 0.0079, 0.0079,
        0.0079, 0.0079, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078,
        0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078,
        0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0078, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077,
        0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0077, 0.0076, 0.0076,
        0.0076, 0.0076, 0.0076, 0.0076, 

In [47]:
print(F.sigmoid(cos_sim))
print(-(F.sigmoid(cos_sim) * torch.log(F.sigmoid(cos_sim))).sum())

tensor([0.6293, 0.6275, 0.6271, 0.6250, 0.6244, 0.6228, 0.6227, 0.6224, 0.6222,
        0.6221, 0.6213, 0.6202, 0.6196, 0.6193, 0.6192, 0.6191, 0.6191, 0.6189,
        0.6189, 0.6188, 0.6187, 0.6185, 0.6185, 0.6184, 0.6183, 0.6182, 0.6181,
        0.6177, 0.6177, 0.6177, 0.6175, 0.6173, 0.6173, 0.6173, 0.6171, 0.6169,
        0.6167, 0.6167, 0.6165, 0.6162, 0.6162, 0.6162, 0.6160, 0.6158, 0.6152,
        0.6147, 0.6147, 0.6145, 0.6143, 0.6137, 0.6136, 0.6132, 0.6132, 0.6132,
        0.6130, 0.6128, 0.6128, 0.6128, 0.6126, 0.6125, 0.6124, 0.6123, 0.6122,
        0.6122, 0.6119, 0.6119, 0.6118, 0.6118, 0.6116, 0.6116, 0.6113, 0.6112,
        0.6111, 0.6109, 0.6109, 0.6106, 0.6106, 0.6106, 0.6105, 0.6105, 0.6104,
        0.6103, 0.6102, 0.6101, 0.6101, 0.6101, 0.6100, 0.6100, 0.6100, 0.6100,
        0.6097, 0.6097, 0.6093, 0.6093, 0.6092, 0.6092, 0.6091, 0.6090, 0.6088,
        0.6087, 0.6086, 0.6086, 0.6086, 0.6086, 0.6085, 0.6085, 0.6084, 0.6084,
        0.6083, 0.6083, 0.6082, 0.6080, 